# Buổi 10 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `lam_sach.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Nhìn lỗ hổng trước khi làm gì khác (chạy lại ô này sau bước 2)

In [ ]:
%matplotlib inline
import warnings

import lam_sach as ls
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
tho = ls.doc_noi_bai()
print(ls.thong_ke_thieu(tho))
print(ls.bao_cao_chat_luong(tho).round(3).to_string(index=False))

bk = ls.doc_bac_kinh()
bang = ls.ty_le_thieu_theo_tram(bk)
fig, ax = plt.subplots(figsize=(11, 3.5))
anh = ax.imshow(bang.T.to_numpy(), aspect="auto", cmap="magma_r", vmin=0, vmax=20)
ax.set_yticks(range(bang.shape[1]), bang.columns, fontsize=7)
ax.set_xlabel("tháng thứ mấy từ 3/2013")
plt.colorbar(anh, ax=ax, label="% giờ thiếu");

## Bước 2 — Ép kiểu và báo cáo chất lượng (mục 4.3)

Sửa `doc_noi_bai`, chạy lại ô bước 1, rồi chạy ô này.

In [ ]:
print("độ phân giải nhiệt độ:", ls.do_phan_giai(tho["temperature"]))
print(ls.doan_tra_hinh(tho).to_string(index=False))
for nguong in (24, 36):
    ket = ls.doan_mac_ket(tho["temperature"], nguong)
    print(f"ngưỡng {nguong} bước: {len(ket)} đoạn đứng yên, {int(ket['so_buoc'].sum())} điểm")
print(ls.doan_mac_ket(tho["temperature"], 36).to_string(index=False))
print("dòng nhiệt độ bị gắn cờ nghi ngờ:", int(ls.co_nghi_ngo(tho, "temperature").sum()))

## Bước 3 — MNAR: mô phỏng và dữ liệu thật (mục 4.2)

In [ ]:
print(ls.mo_phong_mnar())
for tram in ["Dongsi", "Guanyuan", "Wanliu"]:
    print(ls.bang_chung_mnar(bk, tram))

## Bước 4 — Che hai kiểu, so 7 cách điền (mục 4.5)

Sửa `so_sanh_dien` rồi chạy lại ô này (khoảng nửa phút).

In [ ]:
y = ls.luoi_day_du(tho)["temperature"]
hx = ls.doc_hang_xom()["temperature_2m"]
bang_dien = ls.so_sanh_dien(y, hang_xom=hx)
print(ls.bang_xep_hang(bang_dien).round(3).to_string())

## Bước 5 — Pipeline có giới hạn và có cờ (mục 4.6)

Sửa `lam_sach` rồi chạy lại ô này. Cuối cùng: `python lab.py check` trong terminal.

In [ ]:
sach = ls.lam_sach(tho, gioi_han=6, hang_xom=hx)
print(ls.bao_cao_lam_sach(sach))

# Bài kiểm rò rỉ: khoét một lỗ 20 bước, cắt dữ liệu NGAY TRONG lỗ, so phần trước mốc cắt
from tv import ro_ri

nho = tho.iloc[:2000].copy()
nho.iloc[1190:1210, nho.columns.get_loc("temperature")] = np.nan


def ham(df):
    bang = df.set_index("ds")
    bang.index.name = "thoi_gian"
    s = ls.lam_sach(bang, gioi_han=6, hang_xom=hx)
    return pd.DataFrame({"ds": s.index, "sach": s["temperature"].to_numpy()})


dai = nho.reset_index().rename(columns={"thoi_gian": "ds", "temperature": "y"})
dai["ma_temperature"] = nho["ma_temperature"].to_numpy()
print(ro_ri.kiem_ro_ri(ham, dai[["ds", "y", "ma_temperature"]].assign(temperature=dai["y"]),
                       cac_moc_cat=[dai["ds"].iloc[1205]], h=None, cot_id=None, cot_y="y"))